In [2]:
!pip install pygame

  Using cached pygame-2.6.1.tar.gz (14.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  Getting requirements to build wheel did not run successfully.
  exit code: 1
  
  [112 lines of output]
  Skipping Cython compilation
  
  
  WARNING, No "Setup" File Exists, Running "buildconfig/config.py"
  Using WINDOWS configuration...
  
  Making dir :prebuilt_downloads:
  Downloading... https://www.libsdl.org/release/SDL2-devel-2.28.4-VC.zip 25ef9d201ce3fd5f976c37dddedac36bd173975c
  Unzipping :prebuilt_downloads\SDL2-devel-2.28.4-VC.zip:
  Downloading... https://www.libsdl.org/projects/SDL_image/release/SDL2_image-devel-2.0.5-VC.zip 137f86474691f4e12e76e07d58d5920c8d844d5b
  Unzipping :prebuilt_downloads\SDL2_image-devel-2.0.5-VC.zip:
  Downloading... https://github.com/libsdl-org/SDL_ttf/releases/download/release-2.20.1/SDL2_ttf-devel-2.20.1-VC.zip 371606aceba450384428fd2852f73d2f6290b136
  Unzipping :prebuilt_downloads\SDL2_ttf-devel-2.20.1-VC.zip:
  Downloading... https://github.com/libsdl-org/SDL_mixer/releases/download/release-2.6.

In [3]:
import sys
print(sys.version)

3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]


In [2]:
import pygame

pygame 2.6.1 (SDL 2.28.4, Python 3.10.22)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
import pygame
import random
class Game:

    def __init__(self, players, deck):
        self.players = players
        self.deck = deck
        self.current_player = 0
        self.winner = None
        self.game_over = False
        self.turn_actions = {}

        self.start_turn()

    def start_turn(self):
        self.turn_actions = {
            "draw": False,
            "steal": False,
            "draw_2": False,
            "single_discards": 0
        }
    def _get_hand(self, player):
        return player.hand 

    def draw_card(self):
        if self.game_over:
            return False
        player = self.players[self.current_player]
        hand = self._get_hand(player)
        if self.turn_actions["draw"]:
            return False
        if len(hand)>=20:
            return False
        if len(self.deck) == 0:
            return False
        card = self.deck.pop()
        hand.append(card)
        self.turn_actions["draw"] = True
        return True

    def can_steal_card(self, target_index, card_index):  
        if self.game_over:
            return False
        if target_index < 0 or target_index >= len(self.players):
            return False
        current_player = self.players[self.current_player]
        target_player = self.players[target_index]
        
        current_hand = self._get_hand(current_player)
        target_hand = self._get_hand(target_player)

        if target_index == self.current_player:
            return False
        if self.turn_actions["steal"]:
            return False
        if len(current_hand) >= 20:
            return False
        if len(target_hand) == 0:
            return False
        if card_index < 0 or card_index >= len(target_hand):
            return False

        return True

    def steal_card(self, target_index, card_index):

        if not self.can_steal_card(target_index, card_index):
            return False
        current_player = self.players[self.current_player]
        target_player = self.players[target_index]

        current_hand = self._get_hand(current_player)
        target_hand = self._get_hand(target_player)

        stolen_card = target_hand.pop(card_index)
        current_hand.append(stolen_card)

        self.turn_actions["steal"] = True
        
        self.check_winner()

        return True

    def check_winner(self):
        for i,player in enumerate(self.players):
            hand = self._get_hand(player)

            if len(hand) == 0:
                self.winner = i
                self.game_over = True
                return True 

        return False

    def can_discard_card(self, card_index):
        
        if self.game_over:
            return False
            
        player = self.players[self.current_player]
        hand = self._get_hand(player)
       
        if self.turn_actions["single_discards"] >= 2:
            return False

        if card_index < 0 or card_index >= len(hand):
            return False

        return True

    def discard_card(self, card_index):
        
        if not self.can_discard_card(card_index):
            return False
        player = self.players[self.current_player]
        hand = self._get_hand(player)
        
        discarded_card = hand.pop(card_index)

        self.deck.append(discarded_card)

        random.shuffle(self.deck)

        self.turn_actions["single_discards"] += 1

        self.check_winner()

        return True        

    def is_valid_group(self, selected_cards):
        
        if len(selected_cards) < 3:
            return False
        numbers = [card.number for card in selected_cards]

        if len(set(numbers)) == 1:
            return True
            
        sorted_numbers = sorted(numbers)

        for i in range(1, len(sorted_numbers)):
            if sorted_numbers[i] != sorted_numbers[i-1] + 1:
                return False
        colours = {card.colour for card in selected_cards}
        if len(colours)>2:
            return False

        return True                               

    def discard_group(self, selected_cards):
        if self.game_over:
            return False
        player = self.players[self.current_player]
        hand = self._get_hand(player)

        if not self.is_valid_group(selected_cards):
            return False
        remaining_hand = hand.copy()

        for card in selected_cards:
            if card not in remaining_hand:
                return False
            remaining_hand.remove(card)

        for card in selected_cards:
            hand.remove(card)
            
        self.deck.extend(selected_cards)

        random.shuffle(self.deck)
        
        self.check_winner()

        return True

    def end_turn(self):

        if self.game_over:
            return False

        self.current_player = (self.current_player + 1) % len(self.players)

        self.start_turn()

        return True


In [6]:


pygame.init()  # start pygame 


WINDOW_WIDTH = 1920  # screen width
WINDOW_HEIGHT = 1080  # screen height
screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT), pygame.FULLSCREEN | pygame.SCALED)  # fullscreen, fits any monitor
pygame.display.set_caption("Jubilee")  # window title
clock = pygame.time.Clock()  # controls frame rate

#  REAH - Card colours, Card class, deck, Player class, dealing
# Define the card colours using RGB values
CARD_COLOURS = {
    "red": (255, 0, 0),
    "green": (0, 150, 0),
    "yellow": (220, 180, 0),
    "black": (0, 0, 0),
    "blue": (0, 0, 255)
}

# Define the Card class
class Card:
    def __init__(self, colour, number):
        self.colour = colour
        self.number = number

    # Draw a card on the Pygame window
    def draw(self, screen, x, y):
        card_width = 60
        card_height = 90

        # Create the card rectangle
        card_rect = pygame.Rect(x, y, card_width, card_height)

        # Draw the coloured card
        pygame.draw.rect(screen, CARD_COLOURS[self.colour], card_rect)

        # Draw a white outline around the card
        pygame.draw.rect(screen, (255, 255, 255), card_rect, 3)

        # Display the card number in white
        font = pygame.font.SysFont("Arial", 36, bold=True)
        text = font.render(str(self.number), True, (255, 255, 255))

        # Centre the number on the card
        text_rect = text.get_rect(center=card_rect.center)
        screen.blit(text, text_rect)

    # Draw the back of a card
    def draw_back(self, screen, x, y):
        card_width = 60
        card_height = 90
        card_rect = pygame.Rect(x, y, card_width, card_height)

        # Draw a blue card back
        pygame.draw.rect(screen, (30, 60, 150), card_rect)

        # Draw a white outline
        pygame.draw.rect(screen, (255, 255, 255), card_rect, 3)

# Create the deck of 90 cards
colours = ["red", "green", "yellow", "black", "blue"]
numbers = range(1, 10)
cards = []
for colour in colours:
    for number in numbers:
        for copy in range(2):
            card = Card(colour, number)
            cards.append(card)

# Shuffle the deck
random.shuffle(cards)

# Define the Players class
class Player:
    def __init__(self, name):
        self.name = name
        self.hand = []

    # Draw a card from the deck
    def draw_card(self, deck):
        if len(self.hand) < 20 and len(deck) > 0:
            card = deck.pop()
            self.hand.append(card)

# Create three players
player1 = Player("Player 1")
player2 = Player("Player 2")
player3 = Player("Player 3")

players = [player1, player2, player3]

# Deal seven cards to each player
for player in players:
    for i in range(7):
        player.draw_card(cards)

# Check how many cards each player has
for player in players:
    print(player.name, "has", len(player.hand), "cards")



#  YOUSEF - Fonts
title_font = pygame.font.Font(None, 150)  # menu title
button_font = pygame.font.Font(None, 70)  # menu buttons
action_font = pygame.font.Font(None, 40)  # in-game buttons
name_font = pygame.font.Font(None, 46)  # player names
count_font = pygame.font.Font(None, 44)  # card counts
message_font = pygame.font.Font(None, 64)  # turn message
small_font = pygame.font.Font(None, 34)  # small labels


#  YOUSEF - Colours (red, green, blue)
BUTTON_COLOUR = (20, 70, 35)  # dark green
BUTTON_HOVER = (45, 120, 65)  # lighter green
GOLD = (255, 200, 40)  # highlight colour
WHITE = (255, 255, 255)
DARK_TEXT = (25, 25, 25)
PLATE_COLOUR = (10, 40, 20)  # name plate background
TOP_COLOUR = (40, 160, 80)  # gradient top
BOTTOM_COLOUR = (5, 50, 20)  # gradient bottom


#  YOUSEF - Green Background
background = pygame.Surface((WINDOW_WIDTH, WINDOW_HEIGHT))  # blank screen-sized image  (MERGE: Reah's names)
for y in range(WINDOW_HEIGHT):  # each row, top to bottom  (MERGE: Reah's name)
  amount = y / WINDOW_HEIGHT  # 0.0 at top, ~1.0 at bottom  (MERGE: Reah's name)
  red = TOP_COLOUR[0] + (BOTTOM_COLOUR[0] - TOP_COLOUR[0]) * amount  # blend red
  green = TOP_COLOUR[1] + (BOTTOM_COLOUR[1] - TOP_COLOUR[1]) * amount  # blend green
  blue = TOP_COLOUR[2] + (BOTTOM_COLOUR[2] - TOP_COLOUR[2]) * amount  # blend blue
  colour = (int(red), int(green), int(blue))  # whole numbers only
  pygame.draw.line(background, colour, (0, y), (WINDOW_WIDTH, y))  # draw the row  (MERGE: Reah's name)


#  YOUSEF - Card designs
BIG_CARD = (120, 180)  # your hand, deck, discard (width, height)
SMALL_CARD = (90, 135)  # bots' hands (width, height)

CARD_BACK = (90, 20, 30)  # back of card
CARD_BACK_LINE = (150, 40, 50)  # line on the back


def make_card_face(colour, number, card_width, card_height):
  image = pygame.Surface((card_width, card_height), pygame.SRCALPHA)  # see-through image for rounded corners
  card_rect = image.get_rect()  # whole card area
  fill = CARD_COLOURS[colour]  # this card's colour  (MERGE: uses Reah's CARD_COLOURS)
  pygame.draw.rect(image, fill, card_rect, border_radius=12)  # coloured card
  pygame.draw.ellipse(image, WHITE, card_rect.inflate(-card_width // 4, -card_height // 4))  # white oval

  big_font = pygame.font.Font(None, int(card_height * 0.6))  # size fits the card
  corner_font = pygame.font.Font(None, int(card_height * 0.22))
  number_text = str(number)  # number as text

  big_number = big_font.render(number_text, True, fill)  # middle number
  image.blit(big_number, big_number.get_rect(center=card_rect.center))
  corner_number = corner_font.render(number_text, True, WHITE)  # corner numbers
  image.blit(corner_number, corner_number.get_rect(center=(card_width * 0.15, card_height * 0.11)))  # top-left
  image.blit(corner_number, corner_number.get_rect(center=(card_width * 0.85, card_height * 0.89)))  # bottom-right

  pygame.draw.rect(image, WHITE, card_rect, width=4, border_radius=12)  # white edge
  return image


def make_card_back(card_width, card_height):
  image = pygame.Surface((card_width, card_height), pygame.SRCALPHA)  # see-through image
  card_rect = image.get_rect()  # whole card area
  pygame.draw.rect(image, CARD_BACK, card_rect, border_radius=12)  # dark red card
  pygame.draw.rect(image, CARD_BACK_LINE, card_rect.inflate(-20, -20), width=3, border_radius=8)  # inner line
  back_font = pygame.font.Font(None, int(card_height * 0.19))
  word = back_font.render("Jubilee", True, WHITE)  # game name
  image.blit(word, word.get_rect(center=card_rect.center))
  pygame.draw.rect(image, WHITE, card_rect, width=4, border_radius=12)  # white edge
  return image


card_back_image = make_card_back(BIG_CARD[0], BIG_CARD[1])  # big back, for the deck
small_card_back_image = make_card_back(SMALL_CARD[0], SMALL_CARD[1])  # small back, for the bots' hands
card_images = {}  # saved card pictures


def get_card_image(card, size):
  key = (card.colour, card.number, size)  # unique name for this picture
  if key not in card_images:  # not made yet?
    card_images[key] = make_card_face(card.colour, card.number, size[0], size[1])  # make and save it
  return card_images[key]




#  YOUSEF and REAH - Layout

PLAYER_LAYOUT = [  # 0 = you, 1 = Bot 1 (left), 2 = Bot 2 (right)
  {"plate": (60, 950),   "hand_left": 460,  "hand_width": 1100, "hand_y": 860, "size": BIG_CARD},
  {"plate": (60, 200),   "hand_left": 60,   "hand_width": 560,  "hand_y": 300, "size": SMALL_CARD},
  {"plate": (1490, 200), "hand_left": 1300, "hand_width": 560,  "hand_y": 300, "size": SMALL_CARD},
]

DECK_POSITION = (760, 360)  # deck top-left corner
DISCARD_POSITION = (1040, 360)  # discard pile top-left corner

ACTION_NAMES = ["Draw Card", "Take Card", "Discard"]
action_buttons = {}  # button name -> rectangle
button_y = 600  # first button's height
for name in ACTION_NAMES:
  action_buttons[name] = pygame.Rect(1620, button_y, 260, 55)  # button rectangle
  button_y = button_y + 65  # next one lower down
play_for_me_button = pygame.Rect(1620, button_y + 10, 260, 60)  # below the others

start_button = pygame.Rect(0, 0, 400, 100)  # menu Start
start_button.center = (WINDOW_WIDTH // 2, 500)  # (MERGE: Reah's name)
quit_button = pygame.Rect(0, 0, 400, 100)  # menu Quit
quit_button.center = (WINDOW_WIDTH // 2, 650)  # (MERGE: Reah's name)


#  YOUSEF - Drawing things , merged with Reahs card class
def draw_text_centred(text, font, colour, centre):
  text_image = font.render(text, True, colour)  # text as an image
  screen.blit(text_image, text_image.get_rect(center=centre))  # draw it centred


def draw_menu_button(rect, text):
  if rect.collidepoint(pygame.mouse.get_pos()):  # mouse over it?
    pygame.draw.rect(screen, BUTTON_HOVER, rect, border_radius=15)
  else:
    pygame.draw.rect(screen, BUTTON_COLOUR, rect, border_radius=15)
  draw_text_centred(text, button_font, WHITE, rect.center)


def draw_game_button(rect, text, colour, text_colour):
  pygame.draw.rect(screen, colour, rect, border_radius=15)  # no hover, can't be pressed yet
  draw_text_centred(text, action_font, text_colour, rect.center)


def spread_cards(count, area_left, area_width, card_width, gap):
  if count == 0:
    return []
  step = card_width + gap  # space from one card to the next
  if count > 1 and (count - 1) * step + card_width > area_width:  # too wide to fit?
    step = (area_width - card_width) / (count - 1)  # overlap them more
  total_width = (count - 1) * step + card_width  # width of the whole row
  start_x = area_left + (area_width - total_width) / 2  # centre the row
  positions = []
  for i in range(count):
    positions.append(int(start_x + i * step))  # x for each card
  return positions


def find_hovered_card(x_positions, y, size):
  mouse = pygame.mouse.get_pos()  # mouse position
  i = len(x_positions) - 1  # start with the top (last) card
  while i >= 0:
    card_rect = pygame.Rect(x_positions[i], y, size[0], size[1])
    if card_rect.collidepoint(mouse):  # mouse on this card?
      return i
    i = i - 1  # check the card underneath
  return None  # not over any card


def draw_hand(hand, area_left, area_width, y, size, is_you):
  x_positions = spread_cards(len(hand), area_left, area_width, size[0], 12)  # where each card goes
  hovered = None
  if is_you:
    hovered = find_hovered_card(x_positions, y, size)  # which card the mouse is on
  for i in range(len(hand)):
    card_y = y
    if i == hovered:
      card_y = y - 30  # lift the hovered card
    if is_you:
      screen.blit(get_card_image(hand[i], size), (x_positions[i], card_y))  # your cards: face up
    else:
      screen.blit(small_card_back_image, (x_positions[i], card_y))  # bots' cards: face down
    if i == hovered:
      outline = pygame.Rect(x_positions[i], card_y, size[0], size[1]).inflate(8, 8)
      pygame.draw.rect(screen, GOLD, outline, width=5, border_radius=14)  # gold outline


def draw_name_plate(name, card_count, x, y, is_their_turn):
  plate = pygame.Rect(x, y, 300, 60)
  if is_their_turn:
    pygame.draw.rect(screen, GOLD, plate, border_radius=12)  # gold = their turn
    text_colour = DARK_TEXT
  else:
    pygame.draw.rect(screen, PLATE_COLOUR, plate, border_radius=12)
    text_colour = WHITE
  name_image = name_font.render(name, True, text_colour)
  screen.blit(name_image, name_image.get_rect(midleft=(x + 20, plate.centery)))  # name on the left

  badge = pygame.Rect(x + 310, y + 3, 54, 54)  # card count box
  pygame.draw.rect(screen, WHITE, badge, border_radius=8)
  draw_text_centred(str(card_count), count_font, DARK_TEXT, badge.center)


def draw_deck(x, y, cards_left):
  if cards_left == 0:
    pygame.draw.rect(screen, WHITE, (x, y, BIG_CARD[0], BIG_CARD[1]), width=3, border_radius=12)  # empty outline
  else:
    screen.blit(card_back_image, (x + 8, y + 8))  # offset backs look like a pile
    screen.blit(card_back_image, (x + 4, y + 4))
    screen.blit(card_back_image, (x, y))
  draw_text_centred("Deck: " + str(cards_left), small_font, WHITE, (x + BIG_CARD[0] // 2, y + BIG_CARD[1] + 35))


def draw_discard_pile(x, y, pile):
  if len(pile) == 0:
    pygame.draw.rect(screen, WHITE, (x, y, BIG_CARD[0], BIG_CARD[1]), width=3, border_radius=12)  # empty outline
  else:
    screen.blit(get_card_image(pile[-1], BIG_CARD), (x, y))  # top card
  draw_text_centred("Discard", small_font, WHITE, (x + BIG_CARD[0] // 2, y + BIG_CARD[1] + 35))


#  YOUSEF - Screens
def draw_menu():
  screen.blit(background, (0, 0))  # gradient
  draw_text_centred("Jubilee", title_font, WHITE, (WINDOW_WIDTH // 2, 250))  # title  (MERGE: Reah's name)
  draw_menu_button(start_button, "Start Game")
  draw_menu_button(quit_button, "Quit Game")


def draw_game():
  screen.blit(background, (0, 0))  # gradient

  if current_player == 0:  # turn message
    message = "Your turn"
  else:
    message = players[current_player].name + "'s turn"
  draw_text_centred(message, message_font, WHITE, (WINDOW_WIDTH // 2, 70))  # (MERGE: Reah's name)

  draw_deck(DECK_POSITION[0], DECK_POSITION[1], len(cards))  # middle of the table  (MERGE: Reah's deck is called cards)
  draw_discard_pile(DISCARD_POSITION[0], DISCARD_POSITION[1], discard_pile)

  for i in range(len(players)):  # each player  (Reah's players list)
    player = players[i]
    layout = PLAYER_LAYOUT[i]  # where their things go
    draw_name_plate(player.name, len(player.hand), layout["plate"][0], layout["plate"][1], i == current_player)
    draw_hand(player.hand, layout["hand_left"], layout["hand_width"], layout["hand_y"], layout["size"], i == 0)  # i == 0 is you: face up and hoverable

  for name in ACTION_NAMES:  # action buttons (not working yet)
    draw_game_button(action_buttons[name], name, BUTTON_COLOUR, WHITE)
  draw_game_button(play_for_me_button, "Play for Me", GOLD, DARK_TEXT)




# everyone - Main game loop 

current_player = 0  # whose turn: 0 = you

state = "menu"  # "menu" or "game"
running = True  # loop runs while True

while running:
  for event in pygame.event.get():  # everything that happened this frame
    if event.type == pygame.QUIT:  # window closed
      running = False

    elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:  # left click
      if state == "menu":  # clicks only do something on the menu
        if start_button.collidepoint(event.pos):
          state = "game"  # go to the game
        elif quit_button.collidepoint(event.pos):
          running = False  # close the game

      elif state == "game":
        pass  # nothing is clickable in the game yet
        # Game logic stuff foes here

    elif event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:  # Escape key
      if state == "game":
        state = "menu"  # back to menu
      else:
        running = False  # quit from menu

  if state == "menu":  # draw the current screen
    draw_menu()
  elif state == "game":
    draw_game()

  pygame.display.flip()  # show the frame
  clock.tick(60)  # 60 fps

pygame.quit()

 

Player 1 has 7 cards
Player 2 has 7 cards
Player 3 has 7 cards


NameError: name 'discard_pile' is not defined